# 07 · kNN, SVMs and Naive Bayes

After this notebook you can implement k-nearest neighbours from scratch, explain why it needs scaled features and suffers in high dimensions, draw linear and RBF support-vector-machine boundaries and say what `C` and `gamma` do, build a Naive Bayes spam filter on raw text, and pick between these classic models.

**Time:** ~35 min · **Runs:** offline on CPU in < 1 min · **Needs:** [03 · Logistic regression](03_logistic_regression_classification.ipynb), [05 · Overfitting and cross-validation](05_overfitting_regularization_cross_validation.ipynb)

## Why this matters in interviews

- kNN *is* vector search. Every "how does your vector database find the nearest documents?" question starts from brute-force kNN, its O(n·d) cost, distance choice and normalisation.
- SVMs are the standard whiteboard topic for margins, hinge loss and the kernel trick. `C` and `gamma` are classic "what happens if you increase it?" questions.
- A TF-IDF + Naive Bayes (or logistic regression) text classifier is the cheap baseline every LLM classifier should be compared against. It trains in milliseconds.

## Interview questions this notebook answers

| ID | Question |
|----|----------|
| `ve03` | Your vector database has 10 million documents. How does it find the nearest without comparing all 10 million? |
| `ve01` | What is cosine similarity, and what are the alternatives? |
| `ve23` | What is normalisation and when must you do it? |
| `ve20` | What is the curse of dimensionality and does it affect embeddings? |
| `ev17` | How do you evaluate a classification task done by an LLM? |

**Also asked in classical-ML rounds:** how does kNN pick k? why must you scale for kNN and SVMs? what is a support vector? what is the kernel trick? what do `C` and `gamma` control? why is Naive Bayes "naive", and why does it still work for text? what is Laplace smoothing?

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
from sklearn.datasets import load_breast_cancer, load_wine, make_moons
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.neighbors import KNeighborsClassifier, KNeighborsRegressor
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

rng = np.random.default_rng(0)
BLUE, ORANGE, AQUA, YELLOW, GREY = "#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#8a8984"
plt.rcParams.update({"figure.dpi": 100, "axes.spines.top": False, "axes.spines.right": False,
                     "axes.prop_cycle": plt.cycler(color=[BLUE, ORANGE, AQUA, YELLOW, "#4a3aa7"])})
BLUE_ORANGE = LinearSegmentedColormap.from_list("blue_orange", [BLUE, "#f0efec", ORANGE])

def shade(ax, score_fn, X, y, levels=None, title="", n=150):
    """Shade a model's score over a grid (class 0 blue, class 1 orange) and draw the points."""
    xx, yy = np.meshgrid(np.linspace(X[:, 0].min() - 0.4, X[:, 0].max() + 0.4, n),
                         np.linspace(X[:, 1].min() - 0.4, X[:, 1].max() + 0.4, n))
    z = score_fn(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
    lim = np.abs(z).max() if levels is not None else 1
    ax.contourf(xx, yy, z, levels=20, cmap=BLUE_ORANGE, alpha=0.45, vmin=-lim if levels is not None else 0, vmax=lim)
    ax.contour(xx, yy, z, levels=levels if levels is not None else [0.5], colors="black",
               linestyles=["--", "-", "--"] if levels is not None and len(levels) == 3 else "-", linewidths=1.2)
    ax.scatter(X[:, 0], X[:, 1], c=np.where(y == 1, ORANGE, BLUE), s=10, edgecolors="white", linewidths=0.3)
    ax.set(title=title, xticks=[], yticks=[])

## 1. k-nearest neighbours from scratch

**In plain English:** to classify a new point, find the k training points closest to it and let them vote. There is no training step at all: the "model" is the training set. All the work happens at query time.

That is exactly a vector database query: embed the question, find the nearest stored vectors. Brute force costs one distance per stored vector, **O(n·d)** per query, which is why large indexes use approximate methods such as HNSW or IVF (`ve03`, [vector search notebook](../09_rag/03_vector_search_faiss_hnsw.ipynb)).

In [ ]:
def knn_predict(X_train, y_train, X_query, k=5):
    """Brute-force kNN: all pairwise squared distances, take the k smallest, majority vote."""
    d2 = ((X_query[:, None, :] - X_train[None, :, :]) ** 2).sum(axis=-1)    # shape (queries, train)
    nearest = np.argsort(d2, axis=1, kind="stable")[:, :k]
    votes = y_train[nearest]
    return np.array([np.bincount(v).argmax() for v in votes])               # ties -> smallest label, like sklearn

bc = load_breast_cancer()
X_tr, X_te, y_tr, y_te = train_test_split(bc.data, bc.target, test_size=0.3, stratify=bc.target, random_state=0)
scaler = StandardScaler().fit(X_tr)
Xs_tr, Xs_te = scaler.transform(X_tr), scaler.transform(X_te)

ours = knn_predict(Xs_tr, y_tr, Xs_te, k=5)
sk = KNeighborsClassifier(n_neighbors=5).fit(Xs_tr, y_tr).predict(Xs_te)
print(f"our kNN accuracy {np.mean(ours == y_te):.3f}, identical to sklearn on every test row: {bool((ours == sk).all())}")
print(f"work per query: {X_tr.shape[0]} distances x {X_tr.shape[1]} dims = {X_tr.shape[0] * X_tr.shape[1]:,} multiply-adds")
print(f"same brute force over 10M vectors of 768 dims: {10_000_000 * 768:.1e} multiply-adds per query")
assert (ours == sk).all()

### Choosing k

Small k follows every noisy point (high variance); large k smooths everything towards the majority class (high bias). k is picked by cross-validation, usually odd for two classes to avoid ties.

In [ ]:
Xm, ym = make_moons(n_samples=500, noise=0.25, random_state=1)
Xm_tr, Xm_te, ym_tr, ym_te = train_test_split(Xm, ym, test_size=0.4, random_state=0)
ks = [1, 3, 5, 9, 15, 25, 41, 71, 121, 201]
tr_acc = [KNeighborsClassifier(k).fit(Xm_tr, ym_tr).score(Xm_tr, ym_tr) for k in ks]
te_acc = [KNeighborsClassifier(k).fit(Xm_tr, ym_tr).score(Xm_te, ym_te) for k in ks]
best_k = ks[int(np.argmax(te_acc))]

fig, ax = plt.subplots(figsize=(8, 3.4))
ax.plot(ks, tr_acc, "o-", lw=2, label="train accuracy")
ax.plot(ks, te_acc, "o-", lw=2, label="test accuracy")
ax.axvline(best_k, color=GREY, ls="--", label=f"best k = {best_k}")
ax.set(xscale="log", title="kNN on noisy moons: k is the complexity knob (small k = flexible)",
       xlabel="k (log scale)", ylabel="accuracy"); ax.legend(); plt.show()
print(f"k=1: train {tr_acc[0]:.3f} / test {te_acc[0]:.3f}   k={best_k}: test {max(te_acc):.3f}   k=201: test {te_acc[-1]:.3f}")
assert tr_acc[0] == 1.0 and max(te_acc) > te_acc[0] + 0.02 and max(te_acc) > te_acc[-1] + 0.1

### Why scaling matters for kNN

Distance adds up squared differences, so a feature measured in thousands drowns out one measured in tenths. In the wine data, `proline` ranges over about 1,400 units and `hue` over about 1:

In [ ]:
wine = load_wine()
cols = [list(wine.feature_names).index("proline"), list(wine.feature_names).index("hue")]
Xw = wine.data[:, cols]
yw = (wine.target == 1).astype(int)                         # class 1 vs the rest, to keep the picture two-coloured
print("feature ranges (max - min):", {k: round(float(v), 2) for k, v in zip(["proline", "hue"], np.ptp(Xw, axis=0))})
raw_acc = cross_val_score(KNeighborsClassifier(5), Xw, yw, cv=5).mean()
scaled_acc = cross_val_score(make_pipeline(StandardScaler(), KNeighborsClassifier(5)), Xw, yw, cv=5).mean()
print(f"5-fold accuracy: raw {raw_acc:.3f}   standardised {scaled_acc:.3f}")

fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))
for ax, model, name in [(axes[0], KNeighborsClassifier(5), "raw features"),
                        (axes[1], make_pipeline(StandardScaler(), KNeighborsClassifier(5)), "standardised")]:
    model.fit(Xw, yw)
    Xplot = Xw if name == "raw features" else StandardScaler().fit_transform(Xw)
    fn = (lambda Z, m=model: m.predict_proba(Z)[:, 1]) if name == "raw features" else \
         (lambda Z, m=model: m[-1].predict_proba(Z)[:, 1])
    shade(ax, fn, Xplot, yw, title=f"kNN (k=5), {name}")
    ax.set(xlabel="proline" + (" (z-score)" if name != "raw features" else ""), ylabel="hue" + (" (z-score)" if name != "raw features" else ""))
plt.tight_layout(); plt.show()
assert scaled_acc > raw_acc + 0.05

On raw features the boundary is almost a set of vertical stripes: only `proline` matters, because its differences are a thousand times larger. After standardising, both features count. The same applies to SVMs, k-means, PCA and anything built on distances or dot products.

## 2. Distances, normalisation and the curse of dimensionality

**Cosine vs Euclidean (`ve01`, `ve23`).** Cosine similarity compares direction and ignores length. After L2-normalising every vector to length 1, $\lVert a-b\rVert^2 = 2 - 2\cos(a,b)$, so Euclidean distance, dot product and cosine all give **the same ranking**. That is why vector stores normalise once at write time and then use the cheapest metric.

In [ ]:
docs = rng.normal(size=(1000, 64)) * rng.uniform(0.2, 5.0, size=(1000, 1))   # vectors with very different lengths
query = rng.normal(size=64)
cos = docs @ query / (np.linalg.norm(docs, axis=1) * np.linalg.norm(query))
euclid_raw = np.linalg.norm(docs - query, axis=1)
unit_docs, unit_q = docs / np.linalg.norm(docs, axis=1, keepdims=True), query / np.linalg.norm(query)
euclid_unit = np.linalg.norm(unit_docs - unit_q, axis=1)

top_cos, top_raw, top_unit = np.argsort(-cos)[:10], np.argsort(euclid_raw)[:10], np.argsort(euclid_unit)[:10]
print("top-10 overlap, cosine vs raw Euclidean       :", len(set(top_cos) & set(top_raw)), "of 10")
print("top-10 overlap, cosine vs normalised Euclidean:", len(set(top_cos) & set(top_unit)), "of 10")
assert (np.argsort(-cos) == np.argsort(euclid_unit)).all()                # identical full ranking
assert np.allclose(euclid_unit ** 2, 2 - 2 * cos)

**The curse of dimensionality (`ve20`).** Scatter points uniformly in a d-dimensional cube and compare, for a query, the nearest and the farthest neighbour. As d grows the two distances converge, so "nearest" stops meaning much.

In [ ]:
dims = [2, 5, 10, 50, 100, 500, 1000]
contrast = []
for d in dims:
    pts, qs = rng.random((1000, d)), rng.random((50, d))
    dist = np.sqrt(((qs[:, None, :] - pts[None, :, :]) ** 2).sum(-1))
    contrast.append(np.mean((dist.max(axis=1) - dist.min(axis=1)) / dist.min(axis=1)))

fig, ax = plt.subplots(figsize=(7, 3.2))
ax.plot(dims, contrast, "o-", lw=2)
ax.set(xscale="log", yscale="log", title="Uniform random points: nearest and farthest converge",
       xlabel="dimensions d", ylabel="(farthest - nearest) / nearest")
plt.show()
print("relative contrast by dimension:", {d: round(float(c), 3) for d, c in zip(dims, contrast)})
assert contrast[0] > 50 * contrast[-1]

Real embeddings (384 to 3,072 dimensions) escape the worst of this because they are not uniform: they sit on a much lower-dimensional surface shaped by meaning, so neighbourhoods stay informative. [Notebook 08](08_clustering_kmeans_and_pca.ipynb) measures this with PCA. For hand-built features, kNN degrades as you add irrelevant dimensions, so select features first.

## 3. Support vector machines: linear and RBF

**In plain English:** among all lines that separate the classes, an SVM picks the one with the **widest margin**, the fattest street between them. Only the points on or inside the street edges, the **support vectors**, decide where it goes; the others could move and nothing would change.

Technically it minimises the **hinge loss** $\max(0,\ 1 - y\,f(x))$ plus $\tfrac12\lVert w\rVert^2$, with `C` weighting the loss. Swapping the dot product for a **kernel** $k(x, z)$ gives curved boundaries without ever computing the high-dimensional features (the *kernel trick*). The RBF kernel $k(x,z) = e^{-\gamma\lVert x - z\rVert^2}$ is a similarity bump around each support vector.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
svms = {"linear kernel": SVC(kernel="linear", C=1.0), "RBF kernel (gamma=1)": SVC(kernel="rbf", C=1.0, gamma=1.0)}
for ax, (name, svm) in zip(axes, svms.items()):
    svm.fit(Xm_tr, ym_tr)
    shade(ax, svm.decision_function, Xm_tr, ym_tr, levels=[-1, 0, 1],
          title=f"{name}: test acc {svm.score(Xm_te, ym_te):.3f}, {len(svm.support_)} support vectors")
    sv = svm.support_vectors_
    ax.scatter(sv[:, 0], sv[:, 1], s=40, facecolors="none", edgecolors="black", linewidths=0.6)
plt.tight_layout(); plt.show()
assert svms["RBF kernel (gamma=1)"].score(Xm_te, ym_te) > svms["linear kernel"].score(Xm_te, ym_te) + 0.03

Solid line: the boundary (`decision_function` = 0). Dashed lines: the margin edges (±1). Circled points: the support vectors. On noisy, overlapping data many points end up inside the margin, which is why the count is large. `decision_function` is a **margin, not a probability**. To get probabilities, calibrate it on held-out folds: `CalibratedClassifierCV(SVC(), ensemble=False)`. (Older code writes `SVC(probability=True)`; that argument is deprecated since scikit-learn 1.9.)

### The effect of `C` and `gamma`

- **`C`** (the same inverse-regularisation idea as in logistic regression): small `C` tolerates margin violations, giving a wide, smooth street; large `C` punishes every mistake, giving a narrow, wiggly boundary.
- **`gamma`** (RBF width): small `gamma` means each support vector influences a large area (smooth, almost linear); large `gamma` gives tiny bumps around individual points (islands, overfitting).

In [ ]:
Cs, gammas = [0.1, 100], [0.1, 1, 30]
fig, axes = plt.subplots(2, 3, figsize=(11, 4.5))
grid_scores = {}
for i, C in enumerate(Cs):
    for j, g in enumerate(gammas):
        m = SVC(kernel="rbf", C=C, gamma=g).fit(Xm_tr, ym_tr)
        grid_scores[(C, g)] = (m.score(Xm_tr, ym_tr), m.score(Xm_te, ym_te))
        shade(axes[i, j], m.decision_function, Xm_tr, ym_tr, levels=[0], n=100,
              title=f"C={C}, gamma={g}: train {grid_scores[(C, g)][0]:.2f} / test {grid_scores[(C, g)][1]:.2f}")
        axes[i, j].title.set_fontsize(8)
plt.tight_layout(); plt.show()
assert grid_scores[(100, 30)][0] > grid_scores[(100, 30)][1] + 0.05      # big C + big gamma: memorises
assert grid_scores[(0.1, 0.1)][0] < grid_scores[(100, 1)][0]              # small C + small gamma: underfits

Top left (small `C`, small `gamma`) is nearly a straight line and underfits. Bottom right (large `C`, large `gamma`) draws islands around training points: high train accuracy, lower test accuracy. Tune them together, on a log grid, with cross-validation ([notebook 05](05_overfitting_regularization_cross_validation.ipynb) did exactly that).

## 4. Naive Bayes for text: a spam filter

**In plain English:** count how often each word appears in spam and in normal messages ("ham"). For a new message, multiply the per-word evidence together. It is "naive" because it pretends words are independent given the class, which is false, yet the *ranking* it produces is often good, and it trains in one pass over the data.

$$\log P(\text{class}\mid\text{msg}) = \log P(\text{class}) + \sum_{\text{words } w} \text{count}(w)\,\log P(w\mid\text{class}) + \text{const}$$

$P(w\mid\text{class}) = \frac{\text{count}(w, \text{class}) + \alpha}{\text{total words in class} + \alpha V}$, where **Laplace smoothing** ($\alpha = 1$) stops an unseen word from zeroing out a whole class.

In [ ]:
spam = [
    "WIN a FREE iPhone now, click the link to claim your prize",
    "Congratulations! You have been selected for a cash reward, claim now",
    "URGENT: your account is suspended, verify your password at this link",
    "Cheap meds online, no prescription needed, order today",
    "You won a lottery of 1,000,000 dollars, send your bank details to claim",
    "Limited offer: 90% discount on luxury watches, buy now",
    "Earn money from home, 5000 dollars a week, click here",
    "Free entry into our prize draw, text WIN to 80082",
    "Your parcel is waiting, pay the delivery fee at this link now",
    "Hot singles in your area want to meet you, click now",
    "Get a loan approved instantly, no credit check, apply now",
    "Claim your free gift card today, offer expires tonight",
    "Exclusive deal just for you, click the link to unlock your reward",
    "Act now! Double your bitcoin in 24 hours, guaranteed profit",
    "Final notice: your prize is unclaimed, reply YES to receive cash",
]
ham = [
    "Are you free for lunch tomorrow at noon?",
    "Can you send me the slides from today's meeting?",
    "Running ten minutes late, start without me",
    "Happy birthday! Hope you have a great day",
    "The deploy finished, all tests are green",
    "Let's review the pull request after standup",
    "Mum says dinner is at seven on Sunday",
    "Did you book the train tickets for Friday?",
    "Thanks for the help with the report yesterday",
    "The meeting moved to room 4B at 3pm",
    "Can you pick up milk on the way home?",
    "I pushed a fix for the login bug, please take a look",
    "See you at the gym at six",
    "Your order of printer paper will arrive on Tuesday",
    "Let me know when you are free to discuss the budget",
]
texts = spam + ham
labels = np.array([1] * len(spam) + [0] * len(ham))          # 1 = spam
print(f"{len(texts)} messages: {labels.sum()} spam, {len(labels) - labels.sum()} ham")

In [ ]:
vec = CountVectorizer(lowercase=True)
counts = vec.fit_transform(texts).toarray()                  # (messages, vocabulary) word counts
vocab = vec.get_feature_names_out()

def fit_multinomial_nb(C, y, alpha=1.0):
    log_prior = np.log(np.bincount(y) / len(y))
    word_counts = np.vstack([C[y == k].sum(axis=0) for k in (0, 1)]) + alpha      # Laplace smoothing
    log_lik = np.log(word_counts / word_counts.sum(axis=1, keepdims=True))
    return log_prior, log_lik

def nb_log_scores(C, log_prior, log_lik):
    return C @ log_lik.T + log_prior                        # unnormalised log P(class | message)

log_prior, log_lik = fit_multinomial_nb(counts, labels)
sk_nb = MultinomialNB(alpha=1.0).fit(counts, labels)
assert np.allclose(log_lik, sk_nb.feature_log_prob_) and np.allclose(log_prior, sk_nb.class_log_prior_)
assert (nb_log_scores(counts, log_prior, log_lik).argmax(axis=1) == sk_nb.predict(counts)).all()
print(f"vocabulary size: {len(vocab)} words; our NB parameters match MultinomialNB exactly")

evidence = log_lik[1] - log_lik[0]                           # log-odds contribution of each word
order = np.argsort(evidence)
fig, axes = plt.subplots(1, 2, figsize=(11, 3.6))
axes[0].barh(vocab[order[-10:]], evidence[order[-10:]], color=ORANGE); axes[0].set(title="Most spam-indicative words", xlabel="log P(w|spam) - log P(w|ham)")
axes[1].barh(vocab[order[:10]][::-1], -evidence[order[:10]][::-1], color=BLUE); axes[1].set(title="Most ham-indicative words", xlabel="log P(w|ham) - log P(w|spam)")
plt.tight_layout(); plt.show()

With only 30 messages, how well does it generalise? Cross-validate it against the other standard baseline, TF-IDF + logistic regression, then try messages it has never seen.

In [ ]:
cv = StratifiedKFold(5, shuffle=True, random_state=0)
nb_pipe = make_pipeline(CountVectorizer(), MultinomialNB())
lr_pipe = make_pipeline(TfidfVectorizer(), LogisticRegression(C=10))
for name, pipe in [("counts + MultinomialNB", nb_pipe), ("TF-IDF + LogisticRegression", lr_pipe)]:
    s = cross_val_score(pipe, texts, labels, cv=cv)
    print(f"{name:>28}: 5-fold accuracy {s.mean():.2f} ± {s.std():.2f}")

nb_pipe.fit(texts, labels)
new = ["claim your free reward now", "are you free for a quick call about the report?",
       "click the link to verify your bank account", "dinner at my place on Friday?"]
for msg, p in zip(new, nb_pipe.predict_proba(new)[:, 1]):
    print(f"P(spam) = {p:.3f}   {msg}")
assert (nb_pipe.predict(new) == np.array([1, 0, 1, 0])).all()

Notice "free" appears in both spam ("free iPhone") and ham ("are you free for lunch"), and the model still gets the second message right because the other words outweigh it. Two honest caveats: 30 messages is a toy, so the cross-validation spread is wide; and Naive Bayes probabilities are famously **overconfident** (they multiply evidence that is not independent), so use its ranking and calibrate before trusting the numbers ([notebook 04](04_metrics_confusion_matrix_roc_pr.ipynb)).

**The LLM angle (`ev17`):** before you use an LLM to classify tickets, intents or toxicity, fit this baseline. It costs nothing to run, answers in microseconds, and sometimes wins. The LLM has to beat it on your labelled set to earn its latency and cost.

## 5. Which algorithm when

| Model | Training | Prediction | Needs scaling | Strengths | Weaknesses | Reach for it when |
|---|---|---|---|---|---|---|
| kNN | none (store data) | slow: O(n·d) per query | yes | simple, no assumptions, any number of classes | memory, slow queries, high dimensions | small data; similarity search; a baseline for embeddings |
| Linear SVM | fast | fast | yes | strong on high-dimensional sparse data (text) | no probabilities; linear only | text classification at scale (`LinearSVC`) |
| RBF SVM | O(n²) to O(n³) | depends on number of support vectors | yes | flexible boundaries on small/medium data | scales badly past ~50k rows; tuning C and gamma | small tabular or image-feature datasets |
| Naive Bayes | one pass, very fast | very fast | no (counts) | tiny data, text, streaming updates | independence assumption; poor probabilities | a first text baseline; spam; quick prototypes |
| Logistic regression | fast | fast | yes (with a penalty) | calibrated-ish probabilities, interpretable | linear in the features | the default baseline for almost anything |
| Gradient-boosted trees | medium | fast | no | best on tabular data | not for raw text or images | tabular prediction ([06](06_decision_trees_random_forest_boosting.ipynb)) |

## Try it yourself

**1.** kNN *regression*: predict a number as the mean of the k neighbours' targets. Implement it from scratch and check it against `KNeighborsRegressor` on a noisy sine curve.

In [ ]:
# Solution — try it yourself first, then run this
xr = np.sort(rng.uniform(0, 6, 120)); yr = np.sin(xr) + rng.normal(0, 0.2, 120)
xq = np.linspace(0.1, 5.9, 25)
d = np.abs(xq[:, None] - xr[None, :])
ours_r = yr[np.argsort(d, axis=1, kind="stable")[:, :7]].mean(axis=1)
sk_r = KNeighborsRegressor(n_neighbors=7).fit(xr[:, None], yr).predict(xq[:, None])
print("max difference vs sklearn:", np.abs(ours_r - sk_r).max())
assert np.allclose(ours_r, sk_r)

**2.** Show that SVM `decision_function` values are margins, not probabilities: wrap the RBF SVM in `CalibratedClassifierCV(..., method="sigmoid", ensemble=False)` (Platt scaling), fit on the moons training set, and compare margins with calibrated probabilities for five test points.

In [ ]:
# Solution — try it yourself first, then run this
from sklearn.calibration import CalibratedClassifierCV
base = SVC(kernel="rbf", C=1.0, gamma=1.0).fit(Xm_tr, ym_tr)
svm_p = CalibratedClassifierCV(SVC(kernel="rbf", C=1.0, gamma=1.0), method="sigmoid", ensemble=False, cv=5).fit(Xm_tr, ym_tr)
margins, probs = base.decision_function(Xm_te[:5]), svm_p.predict_proba(Xm_te[:5])[:, 1]
for mgn, pr in zip(margins, probs):
    print(f"margin {mgn:+.2f}   ->   Platt-scaled P(class 1) {pr:.2f}")
# margins are unbounded distances to the boundary; predict_proba squashes them with a fitted sigmoid

**3.** Add bigrams to the spam filter (`CountVectorizer(ngram_range=(1, 2))`). Which two-word phrases become the strongest spam signals?

In [ ]:
# Solution — try it yourself first, then run this
vec2 = CountVectorizer(ngram_range=(1, 2))
C2 = vec2.fit_transform(texts)
nb2 = MultinomialNB().fit(C2, labels)
ev2 = nb2.feature_log_prob_[1] - nb2.feature_log_prob_[0]
names2 = vec2.get_feature_names_out()
bigrams = [(names2[i], round(float(ev2[i]), 2)) for i in np.argsort(-ev2) if " " in names2[i]][:6]
print("strongest spam bigrams:", bigrams)

## Say it in an interview

- **kNN in 30 seconds:** "Store the training set; to predict, find the k closest points and vote (or average). No training, O(n·d) per query, sensitive to feature scale and to irrelevant dimensions. Choose k by cross-validation: small k overfits, large k underfits. It is exactly brute-force vector search, which is why vector databases use approximate indexes (HNSW, IVF)."
- **SVM in 30 seconds:** "Find the maximum-margin boundary; only the support vectors matter. The soft margin trades width against violations via C (hinge loss plus L2). Kernels give non-linear boundaries through dot products in an implicit feature space; RBF's gamma sets how local each support vector's influence is. Scale the features; it slows down badly past tens of thousands of rows."
- **Naive Bayes in 30 seconds:** "Bayes' rule plus the assumption that features are independent given the class. For text, count words per class with Laplace smoothing and sum log-probabilities. Absurd assumption, yet a great fast baseline for text; its probabilities are overconfident."
- **Distances:** after L2 normalisation, Euclidean, dot product and cosine rank identically (‖a−b‖² = 2 − 2cos). Normalise documents and queries the same way (`ve23`).
- **Traps:** kNN or SVM on unscaled features; `SVC(probability=True)` in new code (deprecated in scikit-learn 1.9, calibrate instead); kNN with hundreds of irrelevant features; treating SVM margins or NB probabilities as calibrated; an RBF SVM on a million rows; forgetting that kNN's "training" cost is moved to prediction time.

## Next

- [08 · Clustering: k-means and PCA](08_clustering_kmeans_and_pca.ipynb): unsupervised learning on the same distance ideas.
- [Embeddings and similarity](../07_genai_foundations/02_embeddings_and_similarity.ipynb) and [vector search with FAISS and HNSW](../09_rag/03_vector_search_faiss_hnsw.ipynb): kNN at scale.
- [BM25 from scratch](../09_rag/02_bm25_from_scratch.ipynb): the lexical cousin of the word-count features used here.
- Theory: [ml_fundamentals course](../../ml_fundamentals/index.html), [interview bank](../../ai_interview_prep/index.html).